In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from huggingface_hub import HfFileSystem
import polars as pl

fs = HfFileSystem()

number_of_files = 1
event_type = 'ttbar_pu0'
number_of_hf_repo_files= 1000
# Load particles
particles_list = []
for i in range(number_of_files):
    file_path = f"datasets/CERN/ColliderML-Release-1/data/{event_type}_particles/train-{i:05d}-of-{number_of_hf_repo_files:05d}.parquet"
    with fs.open(file_path, "rb") as f:
        particles_list.append(pl.read_parquet(f))
particles = pl.concat(particles_list)

# Load calo_hits
calo_hits_list = []
for i in range(number_of_files):
    file_path = f"datasets/CERN/ColliderML-Release-1/data/{event_type}_calo_hits/train-{i:05d}-of-{number_of_hf_repo_files:05d}.parquet"
    with fs.open(file_path, "rb") as f:
        calo_hits_list.append(pl.read_parquet(f))
calo_hits = pl.concat(calo_hits_list)

# Load tracks
tracks_list = []
for i in range(number_of_files):
    file_path = f"datasets/CERN/ColliderML-Release-1/data/{event_type}_tracks/train-{i:05d}-of-{number_of_hf_repo_files:05d}.parquet"
    with fs.open(file_path, "rb") as f:
        tracks_list.append(pl.read_parquet(f))
tracks = pl.concat(tracks_list)

In [3]:
event_id = 148
particle_id = 882

In [4]:
(particles.lazy()
 .select(['event_id', 'particle_id', 'pdg_id', 'vx', 'vy', 'vz', 'energy'])
 .explode('particle_id', 'pdg_id', 'vx', 'vy', 'vz', 'energy')
.filter((pl.col('event_id')==event_id) & (pl.col('particle_id')==particle_id))   
 ).collect()

event_id,particle_id,pdg_id,vx,vy,vz,energy
u32,u64,i64,f32,f32,f32,f32
148,882,111,-0.003511,-0.010468,125.223251,12.226812


In [6]:
(calo_hits.lazy()
    .select(['event_id', 'contrib_particle_ids','contrib_energies', 'x', 'y', 'z','detector'])
    .explode('contrib_particle_ids', 'contrib_energies', 'x', 'y', 'z', 'detector')
    .explode('contrib_particle_ids', 'contrib_energies') # Double explode if list[list]
    .rename({'contrib_particle_ids': 'particle_id', 'contrib_energies': 'energy_contribution'})
    .filter((pl.col('event_id')==event_id) & (pl.col('particle_id')==particle_id))
    
    ).collect()

event_id,particle_id,energy_contribution,x,y,z,detector
u32,u64,f32,f32,f32,f32,u8
148,882,0.00025,674.331665,-1068.249023,3202.399902,11
148,882,0.000183,697.082153,-1069.865845,3212.5,11


In [7]:
(particles.lazy()
 .select(['event_id','particle_id', 'parent_id', 'pdg_id', 'vx', 'vy', 'vz', 'energy'])
 .explode('particle_id','pdg_id', 'parent_id', 'vx', 'vy', 'vz', 'energy')
.filter((pl.col('event_id')==event_id) & (pl.col('parent_id')==particle_id))
).collect()

event_id,particle_id,parent_id,pdg_id,vx,vy,vz,energy
u32,u64,i64,i64,f32,f32,f32,f32
148,6905,882,22,-0.003053,-0.010893,125.224297,12.144766
148,6906,882,11,68.77462,-98.67469,376.072998,0.012275


In [8]:
(particles.lazy()
 .select(['event_id','particle_id', 'parent_id', 'pdg_id', 'vx', 'vy', 'vz', 'energy'])
 .explode('particle_id','pdg_id', 'parent_id', 'vx', 'vy', 'vz', 'energy')
.filter((pl.col('event_id')==event_id) & (pl.col('parent_id')==6905))
).collect()

event_id,particle_id,parent_id,pdg_id,vx,vy,vz,energy
u32,u64,i64,i64,f32,f32,f32,f32
148,6909,6905,11,258.50827,-239.625351,716.844543,0.862346
148,6910,6905,-11,258.50827,-239.625351,716.844543,11.282419
